In [1]:
import torch
import torch.distributions as TD
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
import numpy as np
from datetime import datetime
import functools


# =========================
# Device
# =========================
enable_cuda = True
device = torch.device('cuda' if torch.cuda.is_available() and enable_cuda else 'cpu')


# =========================
# Parameters
# =========================
param = {
    "test": "type1error",
    "sample_size": 200,
    "batch_size": 128,
    "z_dim": 1,
    "dx": 1,
    "dy": 1,
    "n_test": 100,
    "epochs_num": 200,
    "eps_std": 0.5,
    "dist_z": 'gaussian',
    "alpha_x": 0.75,
    "m_value": 100,
    "k_value": 2,
    "j_value": 1000,
    "noise_dimension": 3,
    "hidden_layer_size": 20,
    "normal_ini": False,
    "preprocess": 'None',
    "G_lr": 1e-3,
    "alpha": 0.1,
    "alpha1": 0.05,
    "set_seeds": 42,
    "using_oracle": False,
    "lambda_1": 1,
    "lambda_2": 0,
    "using_Gen": '1',
    "boor_rv_type": 'gaussian',
    "wgt_decay": 1e-5,
    "lambda_3": 1e-5,
    "drop_out_p": 0.2,
    "M_train": 10,

    # 你的改进
    "lambda_mean": 0.3,
    "mean_samples": 20
}



# =========================
# Data generation: binary version
# =========================
def generate_samples_random(Ax, Ay, size=1000, sType='CI', dx=1, dy=1, dz=1,
                            nstd=0.05, alpha_x=0.05,
                            preprocess="None", dist_z='gaussian'):
    """
    Binary data generator aligned with the author's pasted code.

    Output:
        X, Y, Z are all binary {0,1} tensors.
    """
    if dx != 1 or dy != 1 or dz != 1:
        raise ValueError("This binary version currently supports dx=dy=dz=1 only.")

    num = size

    if sType in ['CI', 'H0', 'type1error']:
        numbers_z = np.random.multinomial(num, [0.5, 0.5], size=1)
        number_z_zeros = numbers_z[0][0]
        number_z_ones = numbers_z[0][1]

        xy_arr = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])

        xy_z_zero_index = np.random.choice(
            a=len(xy_arr), size=number_z_zeros,
            p=[0.25, 0.25, 0.25, 0.25]
        )
        xy_z_one_index = np.random.choice(
            a=len(xy_arr), size=number_z_ones,
            p=[0.25, 0.25, 0.25, 0.25]
        )

        xy_z_zero = xy_arr[xy_z_zero_index]
        xy_z_one = xy_arr[xy_z_one_index]

        xy = np.concatenate((xy_z_zero, xy_z_one), axis=0)

        x = xy[:, 0]
        y = xy[:, 1]
        z = np.concatenate((np.zeros(number_z_zeros), np.ones(number_z_ones)), axis=0)

    elif sType in ['dependent', 'H1', 'power']:
        z = np.random.binomial(1, 0.5, size=num)
        y = np.random.binomial(1, 0.5, size=num)

        delta = np.random.binomial(1, alpha_x, size=num)
        x_indep = np.random.binomial(1, 0.5, size=num)

        x = (1 - delta) * x_indep + delta * y
        x = x.astype(int)

    else:
        raise ValueError("sType must be one of ['CI','H0','type1error','dependent','H1','power']")

    indices = np.random.permutation(num)
    x, y, z = x[indices], y[indices], z[indices]

    X = x.reshape(-1, 1).astype(np.float32)
    Y = y.reshape(-1, 1).astype(np.float32)
    Z = z.reshape(-1, 1).astype(np.float32)

    if preprocess == "normalize":
        pass
    elif preprocess == "scale_Z":
        pass
    elif preprocess == "None":
        pass
    else:
        raise ValueError("preprocess must be one of ['normalize', 'scale_Z', 'None']")

    X = torch.from_numpy(X).float()
    Y = torch.from_numpy(Y).float()
    Z = torch.from_numpy(Z).float()

    return X, Y, Z


# =========================
# Statistic computation
# =========================
def get_p_value_stat_1(boot_num, M, n, gen_x_all_torch, gen_y_all_torch,
                       x_torch, y_torch, z_torch,
                       sigma_w, sigma_u=1, sigma_v=1,
                       boor_rv_type="gaussian"):
    """
    Compute test statistic and bootstrap samples.
    Shapes:
        gen_x_all_torch, gen_y_all_torch: (n, M)
        x_torch, y_torch: (n, 1)
        z_torch: (n, dz)
    """
    w_mx = torch.linalg.vector_norm(
        z_torch.repeat(n, 1, 1) - torch.swapaxes(z_torch.repeat(n, 1, 1), 0, 1),
        ord=1, dim=2
    )
    w_mx = torch.exp(-w_mx / sigma_w)

    u_mx_1 = torch.exp(-torch.abs(y_torch.repeat(1, n) - y_torch.repeat(1, n).T) / sigma_u)
    u_mx_2 = torch.mean(
        torch.exp(
            -torch.abs(gen_y_all_torch.repeat(n, 1, 1) - y_torch.repeat(1, n).reshape(n, n, 1)) / sigma_u
        ),
        dim=2
    )
    u_mx_3 = u_mx_2.T

    gen_y_all_torch_rep = gen_y_all_torch.repeat(n, 1, 1)
    temp_mx = gen_y_all_torch_rep[:, :, 0].T
    sum_mx = torch.mean(
        torch.exp(-torch.abs(gen_y_all_torch_rep - temp_mx.reshape(n, n, 1)) / sigma_u),
        dim=2
    )

    for i in range(1, M):
        temp_mx = gen_y_all_torch_rep[:, :, i].T
        temp_add_mx = torch.mean(
            torch.exp(-torch.abs(gen_y_all_torch_rep - temp_mx.reshape(n, n, 1)) / sigma_u),
            dim=2
        )
        sum_mx = sum_mx + temp_add_mx

    u_mx_4 = 1.0 / M * sum_mx
    u_mx = u_mx_1 - u_mx_2 - u_mx_3 + u_mx_4

    v_mx_1 = torch.exp(-torch.abs(x_torch.repeat(1, n) - x_torch.repeat(1, n).T) / sigma_v)
    v_mx_2 = torch.mean(
        torch.exp(
            -torch.abs(gen_x_all_torch.repeat(n, 1, 1) - x_torch.repeat(1, n).reshape(n, n, 1)) / sigma_v
        ),
        dim=2
    )
    v_mx_3 = v_mx_2.T

    gen_x_all_torch_rep = gen_x_all_torch.repeat(n, 1, 1)
    temp2_mx = gen_x_all_torch_rep[:, :, 0].T
    sum2_mx = torch.mean(
        torch.exp(-torch.abs(gen_x_all_torch_rep - temp2_mx.reshape(n, n, 1)) / sigma_v),
        dim=2
    )

    for i in range(1, M):
        temp2_mx = gen_x_all_torch_rep[:, :, i].T
        temp2_add_mx = torch.mean(
            torch.exp(-torch.abs(gen_x_all_torch_rep - temp2_mx.reshape(n, n, 1)) / sigma_v),
            dim=2
        )
        sum2_mx = sum2_mx + temp2_add_mx

    v_mx_4 = 1.0 / M * sum2_mx
    v_mx = v_mx_1 - v_mx_2 - v_mx_3 + v_mx_4

    FF_mx = u_mx * v_mx * w_mx * (1 - torch.eye(n, device=device))

    stat = 1.0 / (n - 1) * torch.sum(FF_mx).item()

    boottemp = np.array([])

    if boor_rv_type == "rademacher":
        eboot = torch.sign(torch.randn(n, boot_num, device=device))
    elif boor_rv_type == "gaussian":
        eboot = torch.randn(n, boot_num, device=device)
    else:
        raise ValueError("boor_rv_type must be 'rademacher' or 'gaussian'")

    for bb in range(boot_num):
        random_mx = torch.matmul(
            eboot[:, bb].reshape(-1, 1),
            eboot[:, bb].reshape(-1, 1).T
        )
        bootmatrix = FF_mx * random_mx
        stat_boot = 1.0 / (n - 1) * torch.sum(bootmatrix).item()
        boottemp = np.append(boottemp, stat_boot)

    return stat, boottemp


# =========================
# Dataset
# =========================
class DatasetSelect(Dataset):
    def __init__(self, X, Y, Z):
        self.X_real = X
        self.Y_real = Y
        self.Z_real = Z
        self.sample_size = X.shape[0]

    def __len__(self):
        return self.sample_size

    def __getitem__(self, index):
        return self.X_real[index], self.Y_real[index], self.Z_real[index]


class DatasetSelect_GAN(torch.utils.data.Dataset):
    def __init__(self, X, Y, Z, batch_size):
        self.X_real = X
        self.Y_real = Y
        self.Z_real = Z
        self.batch_size = batch_size
        self.sample_size = X.shape[0]

    def __len__(self):
        return self.sample_size

    def __getitem__(self, index):
        return (
            self.X_real[index],
            self.Y_real[index],
            self.Z_real[index],
            self.Z_real[(self.batch_size + index) % self.sample_size]
        )


# =========================
# Noise sampler
# =========================
def sample_noise(sample_size, noise_dimension, noise_type, input_var):
    if noise_type == "normal":
        noise_generator = TD.MultivariateNormal(
            torch.zeros(noise_dimension).to(device),
            input_var * torch.eye(noise_dimension).to(device)
        )
        Z = noise_generator.sample((sample_size,))
    elif noise_type == "unif":
        Z = torch.rand(sample_size, noise_dimension, device=device)
    elif noise_type == "Cauchy":
        Z = TD.Cauchy(
            torch.tensor([0.0], device=device),
            torch.tensor([1.0], device=device)
        ).sample((sample_size, noise_dimension)).squeeze(2)
    else:
        raise ValueError("noise_type must be one of ['normal', 'unif', 'Cauchy']")
    return Z


# =========================
# Generator
# =========================
class Generator(torch.nn.Module):
    def __init__(self, input_dimension, output_dimension, noise_dimension,
                 hidden_layer_size, BN_type, ReLU_coef, drop_out_p,
                 drop_input=False):
        super(Generator, self).__init__()
        self.BN_type = BN_type
        self.ReLU_coef = ReLU_coef
        self.drop_input = drop_input

        self.fc1 = torch.nn.Linear(input_dimension + noise_dimension, hidden_layer_size, bias=True)
        self.fc2 = torch.nn.Linear(hidden_layer_size, hidden_layer_size, bias=True)
        self.fc_last = torch.nn.Linear(hidden_layer_size, output_dimension, bias=True)

        if BN_type:
            self.BN1 = torch.nn.BatchNorm1d(hidden_layer_size, 0.8, affine=False)
            self.BN2 = torch.nn.BatchNorm1d(hidden_layer_size, 0.8, affine=False)

        self.leakyReLU1 = torch.nn.LeakyReLU(ReLU_coef)

        self.drop_out0 = torch.nn.Dropout(p=drop_out_p)
        self.drop_out1 = torch.nn.Dropout(p=drop_out_p)
        self.drop_out2 = torch.nn.Dropout(p=drop_out_p)

        self.sigmoid = torch.nn.Sigmoid()

    def forward(self, x):
        if self.drop_input:
            x = self.drop_out0(x)

        if self.BN_type:
            x = self.drop_out1(self.leakyReLU1(self.BN1(self.fc1(x))))
            x = self.drop_out2(self.leakyReLU1(self.BN2(self.fc2(x))))
        else:
            x = self.drop_out1(self.leakyReLU1(self.fc1(x)))
            x = self.drop_out2(self.leakyReLU1(self.fc2(x)))

        x = self.fc_last(x)

        # Binary probability output
        x = self.sigmoid(x)

        return x


class NonFullyConnected_1(torch.nn.Module):
    def __init__(self, size_in, size_out, m, bias=True):
        super(NonFullyConnected_1, self).__init__()
        self.linear = torch.nn.Linear(m * size_in, m * size_out, bias=bias).to(device)
        self.mask = functools.reduce(
            torch.block_diag,
            [torch.ones(size_out, size_in) for _ in range(m)]
        ).to(device)

    def forward(self, x):
        self.linear.weight.data *= self.mask
        return self.linear(x)


class Generator_2(torch.nn.Module):
    def __init__(self, input_dimension, output_dimension, noise_dimension,
                 hidden_layer_size, BN_type, ReLU_coef,
                 hidden_layer_depth=1, ntargets_k=5):
        super(Generator_2, self).__init__()
        self.input_dimension = input_dimension + noise_dimension
        self.output_dimension = output_dimension
        self.ntargets_k = ntargets_k
        self.hidden_layer_sizes = [hidden_layer_size] * hidden_layer_depth
        self.BN_type = BN_type
        self.leakyrelu = torch.nn.LeakyReLU(ReLU_coef)

        self.linear_layers_from_input = torch.nn.Linear(
            self.input_dimension, ntargets_k * self.hidden_layer_sizes[0]
        )

        self.linear_layers_between = torch.nn.ModuleList([
            NonFullyConnected_1(self.hidden_layer_sizes[0], self.hidden_layer_sizes[0], ntargets_k)
            for _ in range(len(self.hidden_layer_sizes))
        ])

        self.linear8 = torch.nn.Linear(self.hidden_layer_sizes[0] * ntargets_k, self.output_dimension)
        self.sigmoid = torch.nn.Sigmoid()

        if BN_type:
            self.BN1 = torch.nn.BatchNorm1d(hidden_layer_size, 0.8, affine=False)

    def forward(self, input):
        if self.BN_type:
            output = self.linear_layers_from_input(input)
            output = self.leakyrelu(self.BN1(output))
            for linear_layers_between in self.linear_layers_between:
                output = linear_layers_between(output)
                output = self.leakyrelu(self.BN1(output))
        else:
            output = self.linear_layers_from_input(input)
            output = self.leakyrelu(output)
            for linear_layers_between in self.linear_layers_between:
                output = linear_layers_between(output)
                output = self.leakyrelu(output)

        output = self.linear8(output)
        output = self.sigmoid(output)
        return output


# =========================
# Loss
# =========================
def find_loss(y_torch, gen_y_all_torch, z_torch, sigma_w, sigma_u, M):
    n = z_torch.shape[0]

    w_mx = torch.linalg.vector_norm(
        z_torch.repeat(n, 1, 1) - torch.swapaxes(z_torch.repeat(n, 1, 1), 0, 1),
        ord=1, dim=2
    )
    w_mx = torch.exp(-w_mx / sigma_w)

    u_mx_1 = torch.exp(-torch.abs(y_torch.repeat(1, n) - y_torch.repeat(1, n).T) / sigma_u)
    u_mx_2 = torch.mean(
        torch.exp(
            -torch.abs(gen_y_all_torch.repeat(n, 1, 1) - y_torch.repeat(1, n).reshape(n, n, 1)) / sigma_u
        ),
        dim=2
    )
    u_mx_3 = u_mx_2.T

    gen_y_all_torch_rep = gen_y_all_torch.repeat(n, 1, 1)
    temp_mx = gen_y_all_torch_rep[:, :, 0].T
    sum_mx = torch.mean(
        torch.exp(-torch.abs(gen_y_all_torch_rep - temp_mx.reshape(n, n, 1)) / sigma_u),
        dim=2
    )

    for i in range(1, M):
        temp_mx = gen_y_all_torch_rep[:, :, i].T
        temp_add_mx = torch.mean(
            torch.exp(-torch.abs(gen_y_all_torch_rep - temp_mx.reshape(n, n, 1)) / sigma_u),
            dim=2
        )
        sum_mx = sum_mx + temp_add_mx

    u_mx_4 = 1.0 / M * sum_mx
    u_mx = u_mx_1 - u_mx_2 - u_mx_3 + u_mx_4

    FF_mx = u_mx * w_mx * (1 - torch.eye(n, device=device))
    loss = 1.0 / n * torch.sum(FF_mx)

    return loss


# =========================
# Binary sampling helper
# =========================
def sample_binary_from_prob(prob_tensor):
    prob_tensor = torch.clamp(prob_tensor, 1e-6, 1 - 1e-6)
    return torch.bernoulli(prob_tensor)


# =========================
# Training
# =========================
def train_ver3(X, Y, Z, X_test, Y_test, Z_test, M,
               noise_dimension, noise_type, G_lr, hidden_layer_size,
               DataLoader, BN_type, ReLU_coef,
               epochs_num=10, sigma_z=1, sigma_x=1, sigma_y=1,
               normal_ini=False,
               lambda_1=1, lambda_2=1, using_Gen='1', wgt_decay=0,
               lambda_3=0, drop_out_p=0.2, M_train=3,
               lambda_mean=0.0, mean_samples=20):

    input_dimension = Z.shape[1]
    output_dimension_y = Y.shape[1]
    output_dimension_x = X.shape[1]

    if using_Gen == '1':
        G_zy = Generator(
            input_dimension, output_dimension_y, noise_dimension,
            hidden_layer_size, BN_type, ReLU_coef, drop_out_p
        ).to(device)
        G_zy_solver = optim.Adam(G_zy.parameters(), lr=G_lr, betas=(0.5, 0.999), weight_decay=wgt_decay)

        G_zx = Generator(
            input_dimension, output_dimension_x, noise_dimension,
            hidden_layer_size, BN_type, ReLU_coef, drop_out_p
        ).to(device)
        G_zx_solver = optim.Adam(G_zx.parameters(), lr=G_lr, betas=(0.5, 0.999), weight_decay=wgt_decay)

    elif using_Gen == '2':
        G_zy = Generator_2(
            input_dimension, output_dimension_y, noise_dimension,
            hidden_layer_size, BN_type, ReLU_coef
        ).to(device)
        G_zy_solver = optim.Adam(G_zy.parameters(), lr=G_lr, betas=(0.5, 0.999), weight_decay=wgt_decay)

        G_zx = Generator_2(
            input_dimension, output_dimension_x, noise_dimension,
            hidden_layer_size, BN_type, ReLU_coef
        ).to(device)
        G_zx_solver = optim.Adam(G_zx.parameters(), lr=G_lr, betas=(0.5, 0.999), weight_decay=wgt_decay)
    else:
        raise ValueError("using_Gen must be '1' or '2'")

    if normal_ini:
        for p in G_zy.parameters():
            p.data = torch.randn(p.shape, device=device, dtype=torch.float32) / np.sqrt(float(hidden_layer_size * 2))
        for p in G_zx.parameters():
            p.data = torch.randn(p.shape, device=device, dtype=torch.float32) / np.sqrt(float(hidden_layer_size * 2))

    for epoch in range(epochs_num):
        G_zy.train()
        G_zx.train()

        for X_real, Y_real, Z_real, Z_fake in DataLoader:
            X_real = X_real.to(device)
            Y_real = Y_real.to(device)
            Z_real = Z_real.to(device)
            Z_fake = Z_fake.to(device)

            batch_size = Z_real.shape[0]

            # =========================
            # Train G_zy
            # =========================
            G_zy_solver.zero_grad()

            Z_real_repeat_y = Z_real.repeat(M_train, 1)
            Noise_fake_y = sample_noise(
                Z_real_repeat_y.shape[0], noise_dimension, noise_type, input_var=1.0 / 3.0
            ).to(device)

            Y_fake_prob = G_zy(torch.cat((Z_real_repeat_y, Noise_fake_y), dim=1)).to(device)
            Y_fake = Y_fake_prob.reshape(batch_size, M_train)

            l1_regularization_y = 0.0
            for param in G_zy.parameters():
                l1_regularization_y += torch.linalg.vector_norm(param, ord=1)

            mmd_loss_y = lambda_1 * find_loss(
                Y_real, Y_fake, Z_real, sigma_z, sigma_y, M_train
            )

            if lambda_mean > 0:
                # 这里保留你的改进：对同一个 Z 做 Monte Carlo 均值近似
                Z_mean_repeat_y = Z_real.repeat_interleave(mean_samples, dim=0)
                Noise_mean_y = sample_noise(
                    Z_mean_repeat_y.shape[0], noise_dimension, noise_type, input_var=1.0 / 3.0
                ).to(device)

                Y_mean_fake_group = G_zy(torch.cat((Z_mean_repeat_y, Noise_mean_y), dim=1))
                Y_mean_fake = Y_mean_fake_group.reshape(batch_size, mean_samples, output_dimension_y).mean(dim=1)

                mean_loss_y = torch.nn.functional.mse_loss(Y_mean_fake, Y_real)
            else:
                mean_loss_y = 0.0

            g_zy_error = mmd_loss_y + lambda_mean * mean_loss_y + lambda_3 * l1_regularization_y
            g_zy_error.backward()
            torch.nn.utils.clip_grad_norm_(G_zy.parameters(), max_norm=0.5)
            G_zy_solver.step()

            # =========================
            # Train G_zx
            # =========================
            G_zx_solver.zero_grad()

            Z_real_repeat_x = Z_real.repeat(M_train, 1)
            Noise_fake_x = sample_noise(
                Z_real_repeat_x.shape[0], noise_dimension, noise_type, input_var=1.0 / 3.0
            ).to(device)

            X_fake_prob = G_zx(torch.cat((Z_real_repeat_x, Noise_fake_x), dim=1)).to(device)
            X_fake = X_fake_prob.reshape(batch_size, M_train)

            l1_regularization_x = 0.0
            for param in G_zx.parameters():
                l1_regularization_x += torch.linalg.vector_norm(param, ord=1)

            mmd_loss_x = lambda_1 * find_loss(
                X_real, X_fake, Z_real, sigma_z, sigma_x, M_train
            )

            if lambda_mean > 0:
                Z_mean_repeat_x = Z_real.repeat_interleave(mean_samples, dim=0)
                Noise_mean_x = sample_noise(
                    Z_mean_repeat_x.shape[0], noise_dimension, noise_type, input_var=1.0 / 3.0
                ).to(device)

                X_mean_fake_group = G_zx(torch.cat((Z_mean_repeat_x, Noise_mean_x), dim=1))
                X_mean_fake = X_mean_fake_group.reshape(batch_size, mean_samples, output_dimension_x).mean(dim=1)

                mean_loss_x = torch.nn.functional.mse_loss(X_mean_fake, X_real)
            else:
                mean_loss_x = 0.0

            g_zx_error = mmd_loss_x + lambda_mean * mean_loss_x + lambda_3 * l1_regularization_x
            g_zx_error.backward()
            torch.nn.utils.clip_grad_norm_(G_zx.parameters(), max_norm=0.5)
            G_zx_solver.step()

    return G_zy, G_zx

# =========================
# Main experiment for one run
# =========================
def mGAN(Ax, Ay, n=500, z_dim=1, simulation='type1error', batch_size=64, epochs_num=1000,
         nstd=1.0, z_dist='gaussian', x_dims=1, y_dims=1, a_x=0.05, M=500, k=2, boot_num=1000,
         noise_dimension=10, hidden_layer_size=512, normal_ini=False, preprocess='normalize',
         G_lr=1e-5, using_oracle=False, lambda_1=1, lambda_2=1, using_Gen='1',
         boor_rv_type="gaussian", wgt_decay=0, lambda_3=1, drop_out_p=0.2, exp_num=0,
         M_train=3, lambda_mean=0.0, mean_samples=20):


    if simulation == 'type1error':
        sim_x, sim_y, sim_z = generate_samples_random(
            Ax, Ay, size=n, sType='CI', dx=x_dims, dy=y_dims, dz=z_dim,
            nstd=nstd, alpha_x=a_x, dist_z=z_dist, preprocess=preprocess
        )
    elif simulation == 'power':
        sim_x, sim_y, sim_z = generate_samples_random(
            Ax, Ay, size=n, sType='dependent', dx=x_dims, dy=y_dims, dz=z_dim,
            nstd=nstd, alpha_x=a_x, dist_z=z_dist, preprocess=preprocess
        )
    else:
        raise ValueError('Test does not exist.')

    x, y, z = sim_x, sim_y, sim_z

    sigma_w_train = 1.0
    sigma_u_train = 1.0
    sigma_v_train = 1.0

    test_size = int(n / k)
    stat_all = torch.zeros(k, 1)
    boot_temp_all = torch.zeros(k, boot_num)
    cur_k = 0

    for k_fold in range(k):
        k_fold_start = int(n / k * k_fold)
        k_fold_end = int(n / k * (k_fold + 1))

        X_test = x[k_fold_start:k_fold_end]
        Y_test = y[k_fold_start:k_fold_end]
        Z_test = z[k_fold_start:k_fold_end]

        X_train = torch.cat((x[0:k_fold_start], x[k_fold_end:]))
        Y_train = torch.cat((y[0:k_fold_start], y[k_fold_end:]))
        Z_train = torch.cat((z[0:k_fold_start], z[k_fold_end:]))

        if k == 1:
            X_train, Y_train, Z_train = X_test, Y_test, Z_test

        train_xyz = DatasetSelect_GAN(X_train, Y_train, Z_train, batch_size)
        DataLoader_xyz = torch.utils.data.DataLoader(train_xyz, batch_size=batch_size, shuffle=True)

        if not using_oracle:
            G_zy, G_zx = train_ver3(
    X=X_train, Y=Y_train, Z=Z_train, M=M,
    X_test=X_test, Y_test=Y_test, Z_test=Z_test,
    noise_dimension=noise_dimension, noise_type="normal",
    G_lr=G_lr, hidden_layer_size=hidden_layer_size,
    DataLoader=DataLoader_xyz, BN_type=False, ReLU_coef=0.1,
    epochs_num=epochs_num, sigma_z=sigma_w_train, sigma_x=sigma_v_train,
    sigma_y=sigma_u_train, normal_ini=normal_ini, lambda_1=lambda_1,
    lambda_2=lambda_2, using_Gen=using_Gen, wgt_decay=wgt_decay,
    lambda_3=lambda_3, drop_out_p=drop_out_p, M_train=M_train,
    lambda_mean=lambda_mean, mean_samples=mean_samples
)


        dataset_test = DatasetSelect(X_test, Y_test, Z_test)
        dataloader_test = DataLoader(dataset_test, batch_size=1, shuffle=True)

        gen_x_all = torch.zeros(test_size, M)
        gen_y_all = torch.zeros(test_size, M)

        z_all = torch.zeros(test_size, z_dim)
        x_all = torch.zeros(test_size, x_dims)
        y_all = torch.zeros(test_size, y_dims)

        cur_itr = 0

        if using_oracle:
            for i, (x_test, y_test, z_test) in enumerate(dataloader_test):
                z_scalar = int(z_test.item())

                fake_x = torch.bernoulli(0.5 * torch.ones(M)).reshape(1, -1)
                fake_y = torch.bernoulli(0.5 * torch.ones(M)).reshape(1, -1)

                gen_x_all[cur_itr, :] = fake_x.reshape(-1)
                gen_y_all[cur_itr, :] = fake_y.reshape(-1)

                x_all[cur_itr, :] = x_test
                y_all[cur_itr, :] = y_test
                z_all[cur_itr, :] = z_test
                cur_itr += 1
        else:
            G_zx.eval()
            G_zy.eval()

            with torch.no_grad():
                for i, (x_test, y_test, z_test) in enumerate(dataloader_test):
                    z_test_temp = z_test.repeat(M, 1).to(device)

                    noise_x = sample_noise(
                        z_test_temp.size(0), noise_dimension, "normal", input_var=1.0 / 3.0
                    ).to(device)
                    fake_x_prob = G_zx(torch.cat((z_test_temp, noise_x), dim=1))
                    fake_x = sample_binary_from_prob(fake_x_prob).reshape(1, -1)

                    noise_y = sample_noise(
                        z_test_temp.size(0), noise_dimension, "normal", input_var=1.0 / 3.0
                    ).to(device)
                    fake_y_prob = G_zy(torch.cat((z_test_temp, noise_y), dim=1))
                    fake_y = sample_binary_from_prob(fake_y_prob).reshape(1, -1)

                    gen_x_all[cur_itr, :] = fake_x.detach().cpu().reshape(-1)
                    gen_y_all[cur_itr, :] = fake_y.detach().cpu().reshape(-1)
                    x_all[cur_itr, :] = x_test
                    y_all[cur_itr, :] = y_test
                    z_all[cur_itr, :] = z_test
                    cur_itr += 1

        gen_x_all = gen_x_all.to(device)
        gen_y_all = gen_y_all.to(device)

        z_all = Z_test.to(device)
        x_all = X_test.to(device)
        y_all = Y_test.to(device)

        sigma_w = 1.0
        sigma_u = 1.0
        sigma_v = 1.0

        cur_stat, cur_boot_temp = get_p_value_stat_1(
            boot_num, M, test_size,
            gen_x_all.to(device), gen_y_all.to(device),
            x_all.to(device), y_all.to(device), z_all.to(device),
            sigma_w, sigma_u, sigma_v, boor_rv_type
        )

        stat_all[cur_k, :] = cur_stat
        boot_temp_all[cur_k, :] = torch.from_numpy(cur_boot_temp).float()
        cur_k += 1

    return np.mean(torch.mean(boot_temp_all, dim=0).cpu().numpy() > torch.mean(stat_all).item())


# =========================
# Repeated experiment
# =========================
def run_experiment(params):
    test = params["test"]
    sample_size = params["sample_size"]
    batch_size = params["batch_size"]
    z_dim = params["z_dim"]
    dx = params["dx"]
    dy = params["dy"]
    n_test = params["n_test"]
    epochs_num = params["epochs_num"]
    eps_std = params["eps_std"]
    dist_z = params["dist_z"]
    alpha_x = params["alpha_x"]
    m_value = params["m_value"]
    k_value = params["k_value"]
    j_value = params["j_value"]
    noise_dimension = params["noise_dimension"]
    hidden_layer_size = params["hidden_layer_size"]
    normal_ini = params["normal_ini"]
    preprocess = params["preprocess"]
    G_lr = params["G_lr"]
    alpha = params["alpha"]
    alpha1 = params["alpha1"]
    set_seeds = params["set_seeds"]
    using_oracle = params["using_oracle"]
    lambda_1 = params["lambda_1"]
    lambda_2 = params["lambda_2"]
    using_Gen = params["using_Gen"]
    boor_rv_type = params["boor_rv_type"]
    wgt_decay = params["wgt_decay"]
    lambda_3 = params["lambda_3"]
    drop_out_p = params["drop_out_p"]
    M_train = params["M_train"]
    lambda_mean = params["lambda_mean"]
    mean_samples = params["mean_samples"]


    np.random.seed(set_seeds)
    torch.manual_seed(set_seeds)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(set_seeds)

    a_f = torch.rand((z_dim, dx))
    l1_norm_a_f = torch.linalg.vector_norm(a_f, ord=1)
    Ax = a_f / l1_norm_a_f

    a_g = torch.rand((z_dim, dy))
    l1_norm_a_g = torch.linalg.vector_norm(a_g, ord=1)
    Ay = a_g / l1_norm_a_g

    p_values = np.array([])
    test_count = 0

    if test == 'type1error':
        for n_iter in range(n_test):
            start_time = datetime.now()

            p_value = mGAN(
                Ax=Ax, Ay=Ay, n=sample_size, z_dim=z_dim, simulation=test,
                batch_size=batch_size, epochs_num=epochs_num,
                nstd=eps_std, z_dist=dist_z, x_dims=dx, y_dims=dy,
                a_x=alpha_x, M=m_value, k=k_value, boot_num=j_value,
                noise_dimension=noise_dimension, hidden_layer_size=hidden_layer_size,
                normal_ini=normal_ini, preprocess=preprocess, G_lr=G_lr,
                using_oracle=using_oracle, lambda_1=lambda_1, lambda_2=lambda_2,
                using_Gen=using_Gen, boor_rv_type=boor_rv_type, wgt_decay=wgt_decay,
                lambda_3=lambda_3, drop_out_p=drop_out_p, exp_num=n_iter + 1,
                M_train=M_train, lambda_mean=lambda_mean, mean_samples=mean_samples
            )

            test_count += 1
            print("--- The %d'th iteration take %s seconds ---" %
                  (test_count, (datetime.now() - start_time)))

            p_values = np.append(p_values, p_value)

            fp = [pval < alpha for pval in p_values]
            final_result = np.mean(fp)

            fp1 = [pval < alpha1 for pval in p_values]
            final_result1 = np.mean(fp1)

            print('The stat is {}'.format(p_value))
            print('Type 1 error: {} for z dimension {} with significance level {}'.format(
                final_result, z_dim, alpha))
            print('Type 1 error: {} for z dimension {} with significance level {}'.format(
                final_result1, z_dim, alpha1))

    elif test == 'power':
        for n_iter in range(n_test):
            start_time = datetime.now()

            p_value = mGAN(
                Ax=Ax, Ay=Ay, n=sample_size, z_dim=z_dim, simulation=test,
                batch_size=batch_size, epochs_num=epochs_num,
                nstd=eps_std, z_dist=dist_z, x_dims=dx, y_dims=dy,
                a_x=alpha_x, M=m_value, k=k_value, boot_num=j_value,
                noise_dimension=noise_dimension, hidden_layer_size=hidden_layer_size,
                normal_ini=normal_ini, preprocess=preprocess, G_lr=G_lr,
                using_oracle=using_oracle, lambda_1=lambda_1, lambda_2=lambda_2,
                using_Gen=using_Gen, boor_rv_type=boor_rv_type, wgt_decay=wgt_decay,
                lambda_3=lambda_3, drop_out_p=drop_out_p, exp_num=n_iter + 1,
                M_train=M_train
            )

            test_count += 1
            print("--- The %d'th iteration take %s seconds ---" %
                  (test_count, (datetime.now() - start_time)))

            p_values = np.append(p_values, p_value)

            fp = [pval < alpha for pval in p_values]
            final_result = np.mean(fp)

            fp1 = [pval < alpha1 for pval in p_values]
            final_result1 = np.mean(fp1)

            print('The stat is {}'.format(p_value))
            print('Power: {} for z dimension {} with significance level {}'.format(
                final_result, z_dim, alpha))
            print('Power: {} for z dimension {} with significance level {}'.format(
                final_result1, z_dim, alpha1))
    else:
        raise ValueError("test must be 'type1error' or 'power'")

    return p_values


# =========================
# Run
# =========================
run_experiment(param)


--- The 1'th iteration take 0:00:07.619583 seconds ---
The stat is 0.399
Type 1 error: 0.0 for z dimension 1 with significance level 0.1
Type 1 error: 0.0 for z dimension 1 with significance level 0.05
--- The 2'th iteration take 0:00:07.010704 seconds ---
The stat is 0.124
Type 1 error: 0.0 for z dimension 1 with significance level 0.1
Type 1 error: 0.0 for z dimension 1 with significance level 0.05
--- The 3'th iteration take 0:00:06.596040 seconds ---
The stat is 0.792
Type 1 error: 0.0 for z dimension 1 with significance level 0.1
Type 1 error: 0.0 for z dimension 1 with significance level 0.05
--- The 4'th iteration take 0:00:06.879879 seconds ---
The stat is 0.478
Type 1 error: 0.0 for z dimension 1 with significance level 0.1
Type 1 error: 0.0 for z dimension 1 with significance level 0.05
--- The 5'th iteration take 0:00:06.598024 seconds ---
The stat is 0.807
Type 1 error: 0.0 for z dimension 1 with significance level 0.1
Type 1 error: 0.0 for z dimension 1 with significance l

array([0.399, 0.124, 0.792, 0.478, 0.807, 0.646, 0.121, 0.901, 0.503,
       0.266, 0.905, 0.803, 0.836, 0.565, 0.96 , 0.607, 0.257, 0.643,
       0.284, 0.893, 0.715, 0.613, 0.85 , 0.73 , 0.009, 0.44 , 0.015,
       0.979, 0.209, 0.006, 0.028, 0.354, 0.697, 0.974, 0.388, 0.954,
       0.001, 0.176, 0.085, 0.887, 0.253, 0.209, 0.79 , 0.4  , 0.078,
       0.501, 0.79 , 0.523, 0.53 , 0.964, 0.208, 0.208, 0.875, 0.875,
       0.188, 0.756, 0.725, 0.017, 0.161, 0.469, 0.972, 0.472, 0.352,
       0.921, 0.768, 0.173, 0.295, 0.574, 0.854, 0.968, 0.539, 0.934,
       0.894, 0.898, 0.502, 0.6  , 0.327, 0.765, 0.693, 0.506, 0.631,
       0.626, 0.526, 0.543, 0.144, 0.173, 0.49 , 0.393, 0.627, 0.817,
       0.785, 0.76 , 0.842, 0.54 , 0.657, 0.389, 0.741, 0.208, 0.235,
       0.601])

In [2]:
import torch
import torch.distributions as TD
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
import numpy as np
from datetime import datetime
import functools


# =========================
# Device
# =========================
enable_cuda = True
device = torch.device('cuda' if torch.cuda.is_available() and enable_cuda else 'cpu')


# =========================
# Parameters
# =========================
param = {
    "test": "type1error",
    "sample_size": 600,
    "batch_size": 128,
    "z_dim": 1,
    "dx": 1,
    "dy": 1,
    "n_test": 100,
    "epochs_num": 200,
    "eps_std": 0.5,
    "dist_z": 'gaussian',
    "alpha_x": 0.75,
    "m_value": 100,
    "k_value": 2,
    "j_value": 1000,
    "noise_dimension": 3,
    "hidden_layer_size": 20,
    "normal_ini": False,
    "preprocess": 'None',
    "G_lr": 1e-3,
    "alpha": 0.1,
    "alpha1": 0.05,
    "set_seeds": 42,
    "using_oracle": False,
    "lambda_1": 1,
    "lambda_2": 0,
    "using_Gen": '1',
    "boor_rv_type": 'gaussian',
    "wgt_decay": 1e-5,
    "lambda_3": 1e-5,
    "drop_out_p": 0.2,
    "M_train": 10,

    # 你的改进
    "lambda_mean": 0.3,
    "mean_samples": 20
}



# =========================
# Data generation: binary version
# =========================
def generate_samples_random(Ax, Ay, size=1000, sType='CI', dx=1, dy=1, dz=1,
                            nstd=0.05, alpha_x=0.05,
                            preprocess="None", dist_z='gaussian'):
    """
    Binary data generator aligned with the author's pasted code.

    Output:
        X, Y, Z are all binary {0,1} tensors.
    """
    if dx != 1 or dy != 1 or dz != 1:
        raise ValueError("This binary version currently supports dx=dy=dz=1 only.")

    num = size

    if sType in ['CI', 'H0', 'type1error']:
        numbers_z = np.random.multinomial(num, [0.5, 0.5], size=1)
        number_z_zeros = numbers_z[0][0]
        number_z_ones = numbers_z[0][1]

        xy_arr = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])

        xy_z_zero_index = np.random.choice(
            a=len(xy_arr), size=number_z_zeros,
            p=[0.25, 0.25, 0.25, 0.25]
        )
        xy_z_one_index = np.random.choice(
            a=len(xy_arr), size=number_z_ones,
            p=[0.25, 0.25, 0.25, 0.25]
        )

        xy_z_zero = xy_arr[xy_z_zero_index]
        xy_z_one = xy_arr[xy_z_one_index]

        xy = np.concatenate((xy_z_zero, xy_z_one), axis=0)

        x = xy[:, 0]
        y = xy[:, 1]
        z = np.concatenate((np.zeros(number_z_zeros), np.ones(number_z_ones)), axis=0)

    elif sType in ['dependent', 'H1', 'power']:
        z = np.random.binomial(1, 0.5, size=num)
        y = np.random.binomial(1, 0.5, size=num)

        delta = np.random.binomial(1, alpha_x, size=num)
        x_indep = np.random.binomial(1, 0.5, size=num)

        x = (1 - delta) * x_indep + delta * y
        x = x.astype(int)

    else:
        raise ValueError("sType must be one of ['CI','H0','type1error','dependent','H1','power']")

    indices = np.random.permutation(num)
    x, y, z = x[indices], y[indices], z[indices]

    X = x.reshape(-1, 1).astype(np.float32)
    Y = y.reshape(-1, 1).astype(np.float32)
    Z = z.reshape(-1, 1).astype(np.float32)

    if preprocess == "normalize":
        pass
    elif preprocess == "scale_Z":
        pass
    elif preprocess == "None":
        pass
    else:
        raise ValueError("preprocess must be one of ['normalize', 'scale_Z', 'None']")

    X = torch.from_numpy(X).float()
    Y = torch.from_numpy(Y).float()
    Z = torch.from_numpy(Z).float()

    return X, Y, Z


# =========================
# Statistic computation
# =========================
def get_p_value_stat_1(boot_num, M, n, gen_x_all_torch, gen_y_all_torch,
                       x_torch, y_torch, z_torch,
                       sigma_w, sigma_u=1, sigma_v=1,
                       boor_rv_type="gaussian"):
    """
    Compute test statistic and bootstrap samples.
    Shapes:
        gen_x_all_torch, gen_y_all_torch: (n, M)
        x_torch, y_torch: (n, 1)
        z_torch: (n, dz)
    """
    w_mx = torch.linalg.vector_norm(
        z_torch.repeat(n, 1, 1) - torch.swapaxes(z_torch.repeat(n, 1, 1), 0, 1),
        ord=1, dim=2
    )
    w_mx = torch.exp(-w_mx / sigma_w)

    u_mx_1 = torch.exp(-torch.abs(y_torch.repeat(1, n) - y_torch.repeat(1, n).T) / sigma_u)
    u_mx_2 = torch.mean(
        torch.exp(
            -torch.abs(gen_y_all_torch.repeat(n, 1, 1) - y_torch.repeat(1, n).reshape(n, n, 1)) / sigma_u
        ),
        dim=2
    )
    u_mx_3 = u_mx_2.T

    gen_y_all_torch_rep = gen_y_all_torch.repeat(n, 1, 1)
    temp_mx = gen_y_all_torch_rep[:, :, 0].T
    sum_mx = torch.mean(
        torch.exp(-torch.abs(gen_y_all_torch_rep - temp_mx.reshape(n, n, 1)) / sigma_u),
        dim=2
    )

    for i in range(1, M):
        temp_mx = gen_y_all_torch_rep[:, :, i].T
        temp_add_mx = torch.mean(
            torch.exp(-torch.abs(gen_y_all_torch_rep - temp_mx.reshape(n, n, 1)) / sigma_u),
            dim=2
        )
        sum_mx = sum_mx + temp_add_mx

    u_mx_4 = 1.0 / M * sum_mx
    u_mx = u_mx_1 - u_mx_2 - u_mx_3 + u_mx_4

    v_mx_1 = torch.exp(-torch.abs(x_torch.repeat(1, n) - x_torch.repeat(1, n).T) / sigma_v)
    v_mx_2 = torch.mean(
        torch.exp(
            -torch.abs(gen_x_all_torch.repeat(n, 1, 1) - x_torch.repeat(1, n).reshape(n, n, 1)) / sigma_v
        ),
        dim=2
    )
    v_mx_3 = v_mx_2.T

    gen_x_all_torch_rep = gen_x_all_torch.repeat(n, 1, 1)
    temp2_mx = gen_x_all_torch_rep[:, :, 0].T
    sum2_mx = torch.mean(
        torch.exp(-torch.abs(gen_x_all_torch_rep - temp2_mx.reshape(n, n, 1)) / sigma_v),
        dim=2
    )

    for i in range(1, M):
        temp2_mx = gen_x_all_torch_rep[:, :, i].T
        temp2_add_mx = torch.mean(
            torch.exp(-torch.abs(gen_x_all_torch_rep - temp2_mx.reshape(n, n, 1)) / sigma_v),
            dim=2
        )
        sum2_mx = sum2_mx + temp2_add_mx

    v_mx_4 = 1.0 / M * sum2_mx
    v_mx = v_mx_1 - v_mx_2 - v_mx_3 + v_mx_4

    FF_mx = u_mx * v_mx * w_mx * (1 - torch.eye(n, device=device))

    stat = 1.0 / (n - 1) * torch.sum(FF_mx).item()

    boottemp = np.array([])

    if boor_rv_type == "rademacher":
        eboot = torch.sign(torch.randn(n, boot_num, device=device))
    elif boor_rv_type == "gaussian":
        eboot = torch.randn(n, boot_num, device=device)
    else:
        raise ValueError("boor_rv_type must be 'rademacher' or 'gaussian'")

    for bb in range(boot_num):
        random_mx = torch.matmul(
            eboot[:, bb].reshape(-1, 1),
            eboot[:, bb].reshape(-1, 1).T
        )
        bootmatrix = FF_mx * random_mx
        stat_boot = 1.0 / (n - 1) * torch.sum(bootmatrix).item()
        boottemp = np.append(boottemp, stat_boot)

    return stat, boottemp


# =========================
# Dataset
# =========================
class DatasetSelect(Dataset):
    def __init__(self, X, Y, Z):
        self.X_real = X
        self.Y_real = Y
        self.Z_real = Z
        self.sample_size = X.shape[0]

    def __len__(self):
        return self.sample_size

    def __getitem__(self, index):
        return self.X_real[index], self.Y_real[index], self.Z_real[index]


class DatasetSelect_GAN(torch.utils.data.Dataset):
    def __init__(self, X, Y, Z, batch_size):
        self.X_real = X
        self.Y_real = Y
        self.Z_real = Z
        self.batch_size = batch_size
        self.sample_size = X.shape[0]

    def __len__(self):
        return self.sample_size

    def __getitem__(self, index):
        return (
            self.X_real[index],
            self.Y_real[index],
            self.Z_real[index],
            self.Z_real[(self.batch_size + index) % self.sample_size]
        )


# =========================
# Noise sampler
# =========================
def sample_noise(sample_size, noise_dimension, noise_type, input_var):
    if noise_type == "normal":
        noise_generator = TD.MultivariateNormal(
            torch.zeros(noise_dimension).to(device),
            input_var * torch.eye(noise_dimension).to(device)
        )
        Z = noise_generator.sample((sample_size,))
    elif noise_type == "unif":
        Z = torch.rand(sample_size, noise_dimension, device=device)
    elif noise_type == "Cauchy":
        Z = TD.Cauchy(
            torch.tensor([0.0], device=device),
            torch.tensor([1.0], device=device)
        ).sample((sample_size, noise_dimension)).squeeze(2)
    else:
        raise ValueError("noise_type must be one of ['normal', 'unif', 'Cauchy']")
    return Z


# =========================
# Generator
# =========================
class Generator(torch.nn.Module):
    def __init__(self, input_dimension, output_dimension, noise_dimension,
                 hidden_layer_size, BN_type, ReLU_coef, drop_out_p,
                 drop_input=False):
        super(Generator, self).__init__()
        self.BN_type = BN_type
        self.ReLU_coef = ReLU_coef
        self.drop_input = drop_input

        self.fc1 = torch.nn.Linear(input_dimension + noise_dimension, hidden_layer_size, bias=True)
        self.fc2 = torch.nn.Linear(hidden_layer_size, hidden_layer_size, bias=True)
        self.fc_last = torch.nn.Linear(hidden_layer_size, output_dimension, bias=True)

        if BN_type:
            self.BN1 = torch.nn.BatchNorm1d(hidden_layer_size, 0.8, affine=False)
            self.BN2 = torch.nn.BatchNorm1d(hidden_layer_size, 0.8, affine=False)

        self.leakyReLU1 = torch.nn.LeakyReLU(ReLU_coef)

        self.drop_out0 = torch.nn.Dropout(p=drop_out_p)
        self.drop_out1 = torch.nn.Dropout(p=drop_out_p)
        self.drop_out2 = torch.nn.Dropout(p=drop_out_p)

        self.sigmoid = torch.nn.Sigmoid()

    def forward(self, x):
        if self.drop_input:
            x = self.drop_out0(x)

        if self.BN_type:
            x = self.drop_out1(self.leakyReLU1(self.BN1(self.fc1(x))))
            x = self.drop_out2(self.leakyReLU1(self.BN2(self.fc2(x))))
        else:
            x = self.drop_out1(self.leakyReLU1(self.fc1(x)))
            x = self.drop_out2(self.leakyReLU1(self.fc2(x)))

        x = self.fc_last(x)

        # Binary probability output
        x = self.sigmoid(x)

        return x


class NonFullyConnected_1(torch.nn.Module):
    def __init__(self, size_in, size_out, m, bias=True):
        super(NonFullyConnected_1, self).__init__()
        self.linear = torch.nn.Linear(m * size_in, m * size_out, bias=bias).to(device)
        self.mask = functools.reduce(
            torch.block_diag,
            [torch.ones(size_out, size_in) for _ in range(m)]
        ).to(device)

    def forward(self, x):
        self.linear.weight.data *= self.mask
        return self.linear(x)


class Generator_2(torch.nn.Module):
    def __init__(self, input_dimension, output_dimension, noise_dimension,
                 hidden_layer_size, BN_type, ReLU_coef,
                 hidden_layer_depth=1, ntargets_k=5):
        super(Generator_2, self).__init__()
        self.input_dimension = input_dimension + noise_dimension
        self.output_dimension = output_dimension
        self.ntargets_k = ntargets_k
        self.hidden_layer_sizes = [hidden_layer_size] * hidden_layer_depth
        self.BN_type = BN_type
        self.leakyrelu = torch.nn.LeakyReLU(ReLU_coef)

        self.linear_layers_from_input = torch.nn.Linear(
            self.input_dimension, ntargets_k * self.hidden_layer_sizes[0]
        )

        self.linear_layers_between = torch.nn.ModuleList([
            NonFullyConnected_1(self.hidden_layer_sizes[0], self.hidden_layer_sizes[0], ntargets_k)
            for _ in range(len(self.hidden_layer_sizes))
        ])

        self.linear8 = torch.nn.Linear(self.hidden_layer_sizes[0] * ntargets_k, self.output_dimension)
        self.sigmoid = torch.nn.Sigmoid()

        if BN_type:
            self.BN1 = torch.nn.BatchNorm1d(hidden_layer_size, 0.8, affine=False)

    def forward(self, input):
        if self.BN_type:
            output = self.linear_layers_from_input(input)
            output = self.leakyrelu(self.BN1(output))
            for linear_layers_between in self.linear_layers_between:
                output = linear_layers_between(output)
                output = self.leakyrelu(self.BN1(output))
        else:
            output = self.linear_layers_from_input(input)
            output = self.leakyrelu(output)
            for linear_layers_between in self.linear_layers_between:
                output = linear_layers_between(output)
                output = self.leakyrelu(output)

        output = self.linear8(output)
        output = self.sigmoid(output)
        return output


# =========================
# Loss
# =========================
def find_loss(y_torch, gen_y_all_torch, z_torch, sigma_w, sigma_u, M):
    n = z_torch.shape[0]

    w_mx = torch.linalg.vector_norm(
        z_torch.repeat(n, 1, 1) - torch.swapaxes(z_torch.repeat(n, 1, 1), 0, 1),
        ord=1, dim=2
    )
    w_mx = torch.exp(-w_mx / sigma_w)

    u_mx_1 = torch.exp(-torch.abs(y_torch.repeat(1, n) - y_torch.repeat(1, n).T) / sigma_u)
    u_mx_2 = torch.mean(
        torch.exp(
            -torch.abs(gen_y_all_torch.repeat(n, 1, 1) - y_torch.repeat(1, n).reshape(n, n, 1)) / sigma_u
        ),
        dim=2
    )
    u_mx_3 = u_mx_2.T

    gen_y_all_torch_rep = gen_y_all_torch.repeat(n, 1, 1)
    temp_mx = gen_y_all_torch_rep[:, :, 0].T
    sum_mx = torch.mean(
        torch.exp(-torch.abs(gen_y_all_torch_rep - temp_mx.reshape(n, n, 1)) / sigma_u),
        dim=2
    )

    for i in range(1, M):
        temp_mx = gen_y_all_torch_rep[:, :, i].T
        temp_add_mx = torch.mean(
            torch.exp(-torch.abs(gen_y_all_torch_rep - temp_mx.reshape(n, n, 1)) / sigma_u),
            dim=2
        )
        sum_mx = sum_mx + temp_add_mx

    u_mx_4 = 1.0 / M * sum_mx
    u_mx = u_mx_1 - u_mx_2 - u_mx_3 + u_mx_4

    FF_mx = u_mx * w_mx * (1 - torch.eye(n, device=device))
    loss = 1.0 / n * torch.sum(FF_mx)

    return loss


# =========================
# Binary sampling helper
# =========================
def sample_binary_from_prob(prob_tensor):
    prob_tensor = torch.clamp(prob_tensor, 1e-6, 1 - 1e-6)
    return torch.bernoulli(prob_tensor)


# =========================
# Training
# =========================
def train_ver3(X, Y, Z, X_test, Y_test, Z_test, M,
               noise_dimension, noise_type, G_lr, hidden_layer_size,
               DataLoader, BN_type, ReLU_coef,
               epochs_num=10, sigma_z=1, sigma_x=1, sigma_y=1,
               normal_ini=False,
               lambda_1=1, lambda_2=1, using_Gen='1', wgt_decay=0,
               lambda_3=0, drop_out_p=0.2, M_train=3,
               lambda_mean=0.0, mean_samples=20):

    input_dimension = Z.shape[1]
    output_dimension_y = Y.shape[1]
    output_dimension_x = X.shape[1]

    if using_Gen == '1':
        G_zy = Generator(
            input_dimension, output_dimension_y, noise_dimension,
            hidden_layer_size, BN_type, ReLU_coef, drop_out_p
        ).to(device)
        G_zy_solver = optim.Adam(G_zy.parameters(), lr=G_lr, betas=(0.5, 0.999), weight_decay=wgt_decay)

        G_zx = Generator(
            input_dimension, output_dimension_x, noise_dimension,
            hidden_layer_size, BN_type, ReLU_coef, drop_out_p
        ).to(device)
        G_zx_solver = optim.Adam(G_zx.parameters(), lr=G_lr, betas=(0.5, 0.999), weight_decay=wgt_decay)

    elif using_Gen == '2':
        G_zy = Generator_2(
            input_dimension, output_dimension_y, noise_dimension,
            hidden_layer_size, BN_type, ReLU_coef
        ).to(device)
        G_zy_solver = optim.Adam(G_zy.parameters(), lr=G_lr, betas=(0.5, 0.999), weight_decay=wgt_decay)

        G_zx = Generator_2(
            input_dimension, output_dimension_x, noise_dimension,
            hidden_layer_size, BN_type, ReLU_coef
        ).to(device)
        G_zx_solver = optim.Adam(G_zx.parameters(), lr=G_lr, betas=(0.5, 0.999), weight_decay=wgt_decay)
    else:
        raise ValueError("using_Gen must be '1' or '2'")

    if normal_ini:
        for p in G_zy.parameters():
            p.data = torch.randn(p.shape, device=device, dtype=torch.float32) / np.sqrt(float(hidden_layer_size * 2))
        for p in G_zx.parameters():
            p.data = torch.randn(p.shape, device=device, dtype=torch.float32) / np.sqrt(float(hidden_layer_size * 2))

    for epoch in range(epochs_num):
        G_zy.train()
        G_zx.train()

        for X_real, Y_real, Z_real, Z_fake in DataLoader:
            X_real = X_real.to(device)
            Y_real = Y_real.to(device)
            Z_real = Z_real.to(device)
            Z_fake = Z_fake.to(device)

            batch_size = Z_real.shape[0]

            # =========================
            # Train G_zy
            # =========================
            G_zy_solver.zero_grad()

            Z_real_repeat_y = Z_real.repeat(M_train, 1)
            Noise_fake_y = sample_noise(
                Z_real_repeat_y.shape[0], noise_dimension, noise_type, input_var=1.0 / 3.0
            ).to(device)

            Y_fake_prob = G_zy(torch.cat((Z_real_repeat_y, Noise_fake_y), dim=1)).to(device)
            Y_fake = Y_fake_prob.reshape(batch_size, M_train)

            l1_regularization_y = 0.0
            for param in G_zy.parameters():
                l1_regularization_y += torch.linalg.vector_norm(param, ord=1)

            mmd_loss_y = lambda_1 * find_loss(
                Y_real, Y_fake, Z_real, sigma_z, sigma_y, M_train
            )

            if lambda_mean > 0:
                # 这里保留你的改进：对同一个 Z 做 Monte Carlo 均值近似
                Z_mean_repeat_y = Z_real.repeat_interleave(mean_samples, dim=0)
                Noise_mean_y = sample_noise(
                    Z_mean_repeat_y.shape[0], noise_dimension, noise_type, input_var=1.0 / 3.0
                ).to(device)

                Y_mean_fake_group = G_zy(torch.cat((Z_mean_repeat_y, Noise_mean_y), dim=1))
                Y_mean_fake = Y_mean_fake_group.reshape(batch_size, mean_samples, output_dimension_y).mean(dim=1)

                mean_loss_y = torch.nn.functional.mse_loss(Y_mean_fake, Y_real)
            else:
                mean_loss_y = 0.0

            g_zy_error = mmd_loss_y + lambda_mean * mean_loss_y + lambda_3 * l1_regularization_y
            g_zy_error.backward()
            torch.nn.utils.clip_grad_norm_(G_zy.parameters(), max_norm=0.5)
            G_zy_solver.step()

            # =========================
            # Train G_zx
            # =========================
            G_zx_solver.zero_grad()

            Z_real_repeat_x = Z_real.repeat(M_train, 1)
            Noise_fake_x = sample_noise(
                Z_real_repeat_x.shape[0], noise_dimension, noise_type, input_var=1.0 / 3.0
            ).to(device)

            X_fake_prob = G_zx(torch.cat((Z_real_repeat_x, Noise_fake_x), dim=1)).to(device)
            X_fake = X_fake_prob.reshape(batch_size, M_train)

            l1_regularization_x = 0.0
            for param in G_zx.parameters():
                l1_regularization_x += torch.linalg.vector_norm(param, ord=1)

            mmd_loss_x = lambda_1 * find_loss(
                X_real, X_fake, Z_real, sigma_z, sigma_x, M_train
            )

            if lambda_mean > 0:
                Z_mean_repeat_x = Z_real.repeat_interleave(mean_samples, dim=0)
                Noise_mean_x = sample_noise(
                    Z_mean_repeat_x.shape[0], noise_dimension, noise_type, input_var=1.0 / 3.0
                ).to(device)

                X_mean_fake_group = G_zx(torch.cat((Z_mean_repeat_x, Noise_mean_x), dim=1))
                X_mean_fake = X_mean_fake_group.reshape(batch_size, mean_samples, output_dimension_x).mean(dim=1)

                mean_loss_x = torch.nn.functional.mse_loss(X_mean_fake, X_real)
            else:
                mean_loss_x = 0.0

            g_zx_error = mmd_loss_x + lambda_mean * mean_loss_x + lambda_3 * l1_regularization_x
            g_zx_error.backward()
            torch.nn.utils.clip_grad_norm_(G_zx.parameters(), max_norm=0.5)
            G_zx_solver.step()

    return G_zy, G_zx

# =========================
# Main experiment for one run
# =========================
def mGAN(Ax, Ay, n=500, z_dim=1, simulation='type1error', batch_size=64, epochs_num=1000,
         nstd=1.0, z_dist='gaussian', x_dims=1, y_dims=1, a_x=0.05, M=500, k=2, boot_num=1000,
         noise_dimension=10, hidden_layer_size=512, normal_ini=False, preprocess='normalize',
         G_lr=1e-5, using_oracle=False, lambda_1=1, lambda_2=1, using_Gen='1',
         boor_rv_type="gaussian", wgt_decay=0, lambda_3=1, drop_out_p=0.2, exp_num=0,
         M_train=3, lambda_mean=0.0, mean_samples=20):


    if simulation == 'type1error':
        sim_x, sim_y, sim_z = generate_samples_random(
            Ax, Ay, size=n, sType='CI', dx=x_dims, dy=y_dims, dz=z_dim,
            nstd=nstd, alpha_x=a_x, dist_z=z_dist, preprocess=preprocess
        )
    elif simulation == 'power':
        sim_x, sim_y, sim_z = generate_samples_random(
            Ax, Ay, size=n, sType='dependent', dx=x_dims, dy=y_dims, dz=z_dim,
            nstd=nstd, alpha_x=a_x, dist_z=z_dist, preprocess=preprocess
        )
    else:
        raise ValueError('Test does not exist.')

    x, y, z = sim_x, sim_y, sim_z

    sigma_w_train = 1.0
    sigma_u_train = 1.0
    sigma_v_train = 1.0

    test_size = int(n / k)
    stat_all = torch.zeros(k, 1)
    boot_temp_all = torch.zeros(k, boot_num)
    cur_k = 0

    for k_fold in range(k):
        k_fold_start = int(n / k * k_fold)
        k_fold_end = int(n / k * (k_fold + 1))

        X_test = x[k_fold_start:k_fold_end]
        Y_test = y[k_fold_start:k_fold_end]
        Z_test = z[k_fold_start:k_fold_end]

        X_train = torch.cat((x[0:k_fold_start], x[k_fold_end:]))
        Y_train = torch.cat((y[0:k_fold_start], y[k_fold_end:]))
        Z_train = torch.cat((z[0:k_fold_start], z[k_fold_end:]))

        if k == 1:
            X_train, Y_train, Z_train = X_test, Y_test, Z_test

        train_xyz = DatasetSelect_GAN(X_train, Y_train, Z_train, batch_size)
        DataLoader_xyz = torch.utils.data.DataLoader(train_xyz, batch_size=batch_size, shuffle=True)

        if not using_oracle:
            G_zy, G_zx = train_ver3(
    X=X_train, Y=Y_train, Z=Z_train, M=M,
    X_test=X_test, Y_test=Y_test, Z_test=Z_test,
    noise_dimension=noise_dimension, noise_type="normal",
    G_lr=G_lr, hidden_layer_size=hidden_layer_size,
    DataLoader=DataLoader_xyz, BN_type=False, ReLU_coef=0.1,
    epochs_num=epochs_num, sigma_z=sigma_w_train, sigma_x=sigma_v_train,
    sigma_y=sigma_u_train, normal_ini=normal_ini, lambda_1=lambda_1,
    lambda_2=lambda_2, using_Gen=using_Gen, wgt_decay=wgt_decay,
    lambda_3=lambda_3, drop_out_p=drop_out_p, M_train=M_train,
    lambda_mean=lambda_mean, mean_samples=mean_samples
)


        dataset_test = DatasetSelect(X_test, Y_test, Z_test)
        dataloader_test = DataLoader(dataset_test, batch_size=1, shuffle=True)

        gen_x_all = torch.zeros(test_size, M)
        gen_y_all = torch.zeros(test_size, M)

        z_all = torch.zeros(test_size, z_dim)
        x_all = torch.zeros(test_size, x_dims)
        y_all = torch.zeros(test_size, y_dims)

        cur_itr = 0

        if using_oracle:
            for i, (x_test, y_test, z_test) in enumerate(dataloader_test):
                z_scalar = int(z_test.item())

                fake_x = torch.bernoulli(0.5 * torch.ones(M)).reshape(1, -1)
                fake_y = torch.bernoulli(0.5 * torch.ones(M)).reshape(1, -1)

                gen_x_all[cur_itr, :] = fake_x.reshape(-1)
                gen_y_all[cur_itr, :] = fake_y.reshape(-1)

                x_all[cur_itr, :] = x_test
                y_all[cur_itr, :] = y_test
                z_all[cur_itr, :] = z_test
                cur_itr += 1
        else:
            G_zx.eval()
            G_zy.eval()

            with torch.no_grad():
                for i, (x_test, y_test, z_test) in enumerate(dataloader_test):
                    z_test_temp = z_test.repeat(M, 1).to(device)

                    noise_x = sample_noise(
                        z_test_temp.size(0), noise_dimension, "normal", input_var=1.0 / 3.0
                    ).to(device)
                    fake_x_prob = G_zx(torch.cat((z_test_temp, noise_x), dim=1))
                    fake_x = sample_binary_from_prob(fake_x_prob).reshape(1, -1)

                    noise_y = sample_noise(
                        z_test_temp.size(0), noise_dimension, "normal", input_var=1.0 / 3.0
                    ).to(device)
                    fake_y_prob = G_zy(torch.cat((z_test_temp, noise_y), dim=1))
                    fake_y = sample_binary_from_prob(fake_y_prob).reshape(1, -1)

                    gen_x_all[cur_itr, :] = fake_x.detach().cpu().reshape(-1)
                    gen_y_all[cur_itr, :] = fake_y.detach().cpu().reshape(-1)
                    x_all[cur_itr, :] = x_test
                    y_all[cur_itr, :] = y_test
                    z_all[cur_itr, :] = z_test
                    cur_itr += 1

        gen_x_all = gen_x_all.to(device)
        gen_y_all = gen_y_all.to(device)

        z_all = Z_test.to(device)
        x_all = X_test.to(device)
        y_all = Y_test.to(device)

        sigma_w = 1.0
        sigma_u = 1.0
        sigma_v = 1.0

        cur_stat, cur_boot_temp = get_p_value_stat_1(
            boot_num, M, test_size,
            gen_x_all.to(device), gen_y_all.to(device),
            x_all.to(device), y_all.to(device), z_all.to(device),
            sigma_w, sigma_u, sigma_v, boor_rv_type
        )

        stat_all[cur_k, :] = cur_stat
        boot_temp_all[cur_k, :] = torch.from_numpy(cur_boot_temp).float()
        cur_k += 1

    return np.mean(torch.mean(boot_temp_all, dim=0).cpu().numpy() > torch.mean(stat_all).item())


# =========================
# Repeated experiment
# =========================
def run_experiment(params):
    test = params["test"]
    sample_size = params["sample_size"]
    batch_size = params["batch_size"]
    z_dim = params["z_dim"]
    dx = params["dx"]
    dy = params["dy"]
    n_test = params["n_test"]
    epochs_num = params["epochs_num"]
    eps_std = params["eps_std"]
    dist_z = params["dist_z"]
    alpha_x = params["alpha_x"]
    m_value = params["m_value"]
    k_value = params["k_value"]
    j_value = params["j_value"]
    noise_dimension = params["noise_dimension"]
    hidden_layer_size = params["hidden_layer_size"]
    normal_ini = params["normal_ini"]
    preprocess = params["preprocess"]
    G_lr = params["G_lr"]
    alpha = params["alpha"]
    alpha1 = params["alpha1"]
    set_seeds = params["set_seeds"]
    using_oracle = params["using_oracle"]
    lambda_1 = params["lambda_1"]
    lambda_2 = params["lambda_2"]
    using_Gen = params["using_Gen"]
    boor_rv_type = params["boor_rv_type"]
    wgt_decay = params["wgt_decay"]
    lambda_3 = params["lambda_3"]
    drop_out_p = params["drop_out_p"]
    M_train = params["M_train"]
    lambda_mean = params["lambda_mean"]
    mean_samples = params["mean_samples"]


    np.random.seed(set_seeds)
    torch.manual_seed(set_seeds)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(set_seeds)

    a_f = torch.rand((z_dim, dx))
    l1_norm_a_f = torch.linalg.vector_norm(a_f, ord=1)
    Ax = a_f / l1_norm_a_f

    a_g = torch.rand((z_dim, dy))
    l1_norm_a_g = torch.linalg.vector_norm(a_g, ord=1)
    Ay = a_g / l1_norm_a_g

    p_values = np.array([])
    test_count = 0

    if test == 'type1error':
        for n_iter in range(n_test):
            start_time = datetime.now()

            p_value = mGAN(
                Ax=Ax, Ay=Ay, n=sample_size, z_dim=z_dim, simulation=test,
                batch_size=batch_size, epochs_num=epochs_num,
                nstd=eps_std, z_dist=dist_z, x_dims=dx, y_dims=dy,
                a_x=alpha_x, M=m_value, k=k_value, boot_num=j_value,
                noise_dimension=noise_dimension, hidden_layer_size=hidden_layer_size,
                normal_ini=normal_ini, preprocess=preprocess, G_lr=G_lr,
                using_oracle=using_oracle, lambda_1=lambda_1, lambda_2=lambda_2,
                using_Gen=using_Gen, boor_rv_type=boor_rv_type, wgt_decay=wgt_decay,
                lambda_3=lambda_3, drop_out_p=drop_out_p, exp_num=n_iter + 1,
                M_train=M_train, lambda_mean=lambda_mean, mean_samples=mean_samples
            )

            test_count += 1
            print("--- The %d'th iteration take %s seconds ---" %
                  (test_count, (datetime.now() - start_time)))

            p_values = np.append(p_values, p_value)

            fp = [pval < alpha for pval in p_values]
            final_result = np.mean(fp)

            fp1 = [pval < alpha1 for pval in p_values]
            final_result1 = np.mean(fp1)

            print('The stat is {}'.format(p_value))
            print('Type 1 error: {} for z dimension {} with significance level {}'.format(
                final_result, z_dim, alpha))
            print('Type 1 error: {} for z dimension {} with significance level {}'.format(
                final_result1, z_dim, alpha1))

    elif test == 'power':
        for n_iter in range(n_test):
            start_time = datetime.now()

            p_value = mGAN(
                Ax=Ax, Ay=Ay, n=sample_size, z_dim=z_dim, simulation=test,
                batch_size=batch_size, epochs_num=epochs_num,
                nstd=eps_std, z_dist=dist_z, x_dims=dx, y_dims=dy,
                a_x=alpha_x, M=m_value, k=k_value, boot_num=j_value,
                noise_dimension=noise_dimension, hidden_layer_size=hidden_layer_size,
                normal_ini=normal_ini, preprocess=preprocess, G_lr=G_lr,
                using_oracle=using_oracle, lambda_1=lambda_1, lambda_2=lambda_2,
                using_Gen=using_Gen, boor_rv_type=boor_rv_type, wgt_decay=wgt_decay,
                lambda_3=lambda_3, drop_out_p=drop_out_p, exp_num=n_iter + 1,
                M_train=M_train
            )

            test_count += 1
            print("--- The %d'th iteration take %s seconds ---" %
                  (test_count, (datetime.now() - start_time)))

            p_values = np.append(p_values, p_value)

            fp = [pval < alpha for pval in p_values]
            final_result = np.mean(fp)

            fp1 = [pval < alpha1 for pval in p_values]
            final_result1 = np.mean(fp1)

            print('The stat is {}'.format(p_value))
            print('Power: {} for z dimension {} with significance level {}'.format(
                final_result, z_dim, alpha))
            print('Power: {} for z dimension {} with significance level {}'.format(
                final_result1, z_dim, alpha1))
    else:
        raise ValueError("test must be 'type1error' or 'power'")

    return p_values


# =========================
# Run
# =========================
run_experiment(param)


--- The 1'th iteration take 0:00:22.794787 seconds ---
The stat is 0.149
Type 1 error: 0.0 for z dimension 1 with significance level 0.1
Type 1 error: 0.0 for z dimension 1 with significance level 0.05
--- The 2'th iteration take 0:00:22.400714 seconds ---
The stat is 0.573
Type 1 error: 0.0 for z dimension 1 with significance level 0.1
Type 1 error: 0.0 for z dimension 1 with significance level 0.05
--- The 3'th iteration take 0:00:22.461433 seconds ---
The stat is 0.403
Type 1 error: 0.0 for z dimension 1 with significance level 0.1
Type 1 error: 0.0 for z dimension 1 with significance level 0.05
--- The 4'th iteration take 0:00:22.365360 seconds ---
The stat is 0.445
Type 1 error: 0.0 for z dimension 1 with significance level 0.1
Type 1 error: 0.0 for z dimension 1 with significance level 0.05
--- The 5'th iteration take 0:00:22.421838 seconds ---
The stat is 0.796
Type 1 error: 0.0 for z dimension 1 with significance level 0.1
Type 1 error: 0.0 for z dimension 1 with significance l

array([0.149, 0.573, 0.403, 0.445, 0.796, 0.969, 0.986, 0.753, 0.506,
       0.595, 0.924, 0.22 , 0.778, 0.13 , 0.043, 0.473, 0.597, 0.499,
       0.666, 0.088, 0.301, 0.786, 0.423, 0.549, 0.285, 0.397, 0.201,
       0.45 , 0.467, 0.168, 0.543, 0.197, 0.719, 0.817, 0.181, 0.359,
       0.829, 0.15 , 0.697, 0.836, 0.421, 0.356, 0.168, 0.274, 0.485,
       0.763, 0.134, 0.499, 0.133, 0.34 , 0.952, 0.155, 0.898, 0.985,
       0.087, 0.734, 0.192, 0.524, 0.104, 0.501, 0.215, 0.614, 0.041,
       0.206, 0.627, 0.835, 0.007, 0.386, 0.983, 0.513, 0.288, 0.292,
       0.074, 0.234, 0.924, 0.157, 0.387, 0.823, 0.838, 0.613, 0.762,
       0.333, 0.648, 0.988, 0.143, 0.652, 0.658, 0.345, 0.855, 0.375,
       0.887, 0.293, 0.135, 0.009, 0.727, 0.057, 0.997, 0.623, 0.61 ,
       0.329])